# 0. Title and Overview

# Promptable Biomedical Lesion Segmentation with MedSAM-Inspired Reasoning

**Subtitle:** A Colab-ready, self-learning notebook for understanding prompt-guided medical image segmentation with lightweight synthetic training.

Biomedical segmentation asks a deceptively simple question: which pixels belong to a biologically or clinically meaningful structure? In practice, this task is difficult because lesions can be small, low contrast, noisy, heterogeneous, or ambiguous. Promptable segmentation foundation models, such as SAM and MedSAM-style systems, change the workflow from purely automatic segmentation to human-in-the-loop segmentation: a user supplies a prompt, such as a box or point, and the model returns a mask for the requested object.

This notebook teaches the core idea without requiring large downloads, clinical data access, or expensive compute. You will build a tiny prompt-guided segmentation system that takes three inputs: a synthetic biomedical image, a bounding-box prompt, and a point-like prompt channel. The model is inspired by the reasoning pattern behind MedSAM, but it is intentionally small enough for Google Colab Free Tier.

You will implement:

- a deterministic synthetic lesion image generator,
- a prompt representation using box and point channels,
- a classical prompt-aware baseline,
- a tiny prompt-guided segmentation network,
- a lightweight training loop using `FAST_MODE = True`,
- quantitative biomedical segmentation metrics,
- visualization, ablation, and failure-mode analysis.

You will lightly train a tiny model for a few epochs on synthetic lesion images. You will visualize predictions, prompts, error maps, uncertainty maps, training curves, and stress-test results. You will measure Dice, IoU, sensitivity, and specificity.

This notebook is designed to run from top to bottom in **Google Colab**. It follows **Concept first, code second**: every code cell is preceded by an explanation of the concept, intuition, and expected result.

## Table of Contents

0. Colab Setup and Runtime Check  
2. Learning Goals  
3. Background and Motivation  
4. Mathematical Core  
5. Synthetic Dataset Generator  
6. Baseline Method  
7. Lightweight Learned or Research-Inspired Method  
8. Lightweight Training Loop  
9. Evaluation Metrics  
10. Visualization and Interpretation  
11. Ablation or Stress Test  
12. Failure Modes and Limitations  
13. Optional Full-Scale Training Resources  
14. Research Extension Mini-Project  
15. Exercises and Reflection Questions  
16. Summary and Further Reading

## Key References and URLs

- MedSAM official repository: https://github.com/bowang-lab/MedSAM
- MedSAM paper, *Segment Anything in Medical Images*: https://arxiv.org/abs/2304.12306
- Nature Communications version: https://www.nature.com/articles/s41467-024-44824-z
- Segment Anything official repository: https://github.com/facebookresearch/segment-anything
- Segment Anything paper: https://arxiv.org/abs/2304.02643
- MONAI documentation: https://docs.monai.io/
- Medical Segmentation Decathlon: http://medicaldecathlon.com/
- nnU-Net: https://github.com/MIC-DKFZ/nnUNet

**Important note:** This notebook does not train MedSAM. Full MedSAM-scale training requires large medical datasets and substantial compute. Instead, this notebook builds a tiny prompt-guided model that captures the educational logic of promptable segmentation.


## 0. Colab Setup and Runtime Check

Runtime setup matters because biomedical computer vision experiments often behave differently on CPU and GPU, and because reproducibility depends on random seeds. The next cell imports only lightweight packages that are typically available in Colab, sets deterministic seeds, selects a GPU automatically when available, and defines compact plotting defaults. The expected result is a short runtime summary that tells you whether the notebook is using CPU or CUDA.


In [ ]:
import os
import math
import random
import platform
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, binary_opening, binary_closing

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

try:
    import pandas as pd
except Exception:
    pd = None

SEED = 42
FAST_MODE = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# Keep CPU execution predictable and lightweight for shared notebook runtimes.
torch.set_num_threads(min(2, os.cpu_count() or 1))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = False
plt.rcParams["image.cmap"] = "gray"

print("Using device:", device)
print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("FAST_MODE:", FAST_MODE)


# 2. Learning Goals

By the end of this notebook, you should be able to:

1. Explain why prompt-guided segmentation addresses a key limitation of fully automatic biomedical segmentation.
2. Describe how a box prompt and point prompt can condition a segmentation model.
3. Implement a deterministic synthetic lesion dataset for controlled experiments.
4. Implement Dice loss, Dice score, IoU, sensitivity, and specificity.
5. Compare a classical prompt-aware thresholding baseline against a tiny learned prompt-guided model.
6. Train a very small segmentation network for a few epochs on synthetic data.
7. Visualize predictions, prompts, uncertainty, and error maps.
8. Analyze prompt-quality stress tests and failure cases as a researcher would.


# 3. Background and Motivation

In clinical and biomedical research, segmentation is used to delineate organs, tumors, lesions, cells, nuclei, vessels, plaques, colonies, and other structures. A segmentation mask may support measurement, diagnosis, treatment planning, biological quantification, or robot-assisted laboratory workflows.

Naive methods often fail because biomedical images can have weak edges, nonuniform intensity, overlapping structures, sensor noise, blur, artifacts, and domain shifts across scanners, stains, microscopes, protocols, or laboratories. A fixed threshold may work on one image and fail on the next.

Prompt-guided segmentation changes the problem. Instead of asking a model to find every possible object, a user tells the model what object to segment. This makes the task more interactive and often more robust. In a MedSAM-style workflow, a prompt such as a bounding box focuses the model on the target anatomy or lesion. The model must still infer the boundary from image evidence.

The method assumes that the prompt roughly localizes the target and that the image contains enough information to distinguish foreground from background. In practice, students should watch for low contrast, tiny targets, inaccurate prompts, severe noise, and out-of-distribution anatomy or imaging style.

The next cell creates one toy lesion image. You should see a synthetic medical-image-like slice, a ground-truth lesion mask, and a prompt box. This example is not meant to be realistic clinical data; it is meant to isolate the reasoning pattern.


In [ ]:
def make_prompt_channels(image_size, box, center_yx, click_sigma=3.0):
    """Create a box channel and a Gaussian click channel for a prompt."""
    y0, x0, y1, x1 = box
    box_channel = np.zeros((image_size, image_size), dtype=np.float32)
    box_channel[y0:y1 + 1, x0:x1 + 1] = 1.0

    yy, xx = np.mgrid[0:image_size, 0:image_size]
    cy, cx = center_yx
    click_channel = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * click_sigma ** 2))
    click_channel = click_channel.astype(np.float32)
    return box_channel, click_channel


def synthetic_lesion_sample(
    image_size=64,
    seed=0,
    prompt_jitter=3,
    contrast=None,
    noise_sigma=None,
    lesion_scale=None,
    irregularity=0.12,
):
    """Generate one deterministic synthetic lesion image, mask, and prompt."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:image_size, 0:image_size]

    if lesion_scale is None:
        lesion_scale = rng.uniform(0.12, 0.25)
    if contrast is None:
        contrast = rng.uniform(0.35, 0.75)
    if noise_sigma is None:
        noise_sigma = rng.uniform(0.04, 0.10)

    cy = rng.uniform(0.30 * image_size, 0.70 * image_size)
    cx = rng.uniform(0.30 * image_size, 0.70 * image_size)
    a = rng.uniform(0.7, 1.3) * lesion_scale * image_size
    b = rng.uniform(0.7, 1.3) * lesion_scale * image_size
    angle = rng.uniform(0, np.pi)

    x = xx - cx
    y = yy - cy
    xr = x * np.cos(angle) + y * np.sin(angle)
    yr = -x * np.sin(angle) + y * np.cos(angle)
    radius = np.sqrt((xr / a) ** 2 + (yr / b) ** 2)
    theta = np.arctan2(yr / max(b, 1e-6), xr / max(a, 1e-6))
    boundary = 1.0 + irregularity * np.sin(3 * theta + rng.uniform(0, 2 * np.pi))
    boundary += 0.05 * np.sin(7 * theta + rng.uniform(0, 2 * np.pi))
    mask = (radius <= boundary).astype(np.float32)

    background = rng.normal(0.0, 1.0, size=(image_size, image_size)).astype(np.float32)
    background = gaussian_filter(background, sigma=rng.uniform(2.0, 5.0))
    background = (background - background.min()) / (background.max() - background.min() + 1e-8)

    gradient = (xx / image_size) * rng.uniform(-0.15, 0.15) + (yy / image_size) * rng.uniform(-0.15, 0.15)
    lesion_texture = gaussian_filter(rng.normal(size=(image_size, image_size)).astype(np.float32), sigma=1.0)
    lesion_texture = 0.08 * lesion_texture * mask

    image = 0.45 * background + gradient + contrast * gaussian_filter(mask, sigma=0.8) + lesion_texture
    image += rng.normal(0.0, noise_sigma, size=(image_size, image_size)).astype(np.float32)
    image = gaussian_filter(image, sigma=rng.uniform(0.0, 0.7))
    image = (image - image.min()) / (image.max() - image.min() + 1e-8)
    image = image.astype(np.float32)

    coords = np.argwhere(mask > 0.5)
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0)

    if prompt_jitter > 0:
        deltas = rng.integers(-prompt_jitter, prompt_jitter + 1, size=4)
        y0 = int(np.clip(y0 + deltas[0], 0, image_size - 2))
        x0 = int(np.clip(x0 + deltas[1], 0, image_size - 2))
        y1 = int(np.clip(y1 + deltas[2], y0 + 1, image_size - 1))
        x1 = int(np.clip(x1 + deltas[3], x0 + 1, image_size - 1))

    box = (int(y0), int(x0), int(y1), int(x1))
    center_yx = ((y0 + y1) / 2.0, (x0 + x1) / 2.0)
    box_channel, click_channel = make_prompt_channels(image_size, box, center_yx)

    return {
        "image": image,
        "mask": mask.astype(np.float32),
        "box": box,
        "box_channel": box_channel,
        "click_channel": click_channel,
        "contrast": float(contrast),
        "noise_sigma": float(noise_sigma),
    }


def draw_box(ax, box, color="yellow", linewidth=2):
    """Draw a prompt box on an axis."""
    y0, x0, y1, x1 = box
    ax.plot([x0, x1, x1, x0, x0], [y0, y0, y1, y1, y0], color=color, linewidth=linewidth)

sample = synthetic_lesion_sample(image_size=64, seed=SEED)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(sample["image"], cmap="gray")
draw_box(axes[0], sample["box"])
axes[0].set_title("Synthetic image with prompt box")
axes[1].imshow(sample["mask"], cmap="gray")
axes[1].set_title("Ground-truth lesion mask")
axes[2].imshow(sample["click_channel"], cmap="gray")
axes[2].set_title("Point-like prompt channel")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


# 4. Mathematical Core

Segmentation can be viewed as binary pixel classification. For each pixel $i$, the ground-truth label is $y_i \in \{0,1\}$ and the model predicts a probability $p_i \in [0,1]$.

A central metric in biomedical segmentation is the Dice score. It measures overlap between the predicted mask and the ground-truth mask:

$$
Dice(y,p) = \frac{2\sum_i y_i p_i + \epsilon}{\sum_i y_i + \sum_i p_i + \epsilon}
$$

Here, $\epsilon$ is a small constant that prevents division by zero. Dice is high when the predicted and true foreground regions overlap strongly. Dice is often preferred for biomedical tasks because foreground objects may occupy only a small fraction of the image.

A common training objective is Dice loss:

$$
L_{Dice}(y,p) = 1 - Dice(y,p)
$$

We will also use binary cross entropy with logits because it gives stable pixel-wise gradients:

$$
L = L_{BCE} + L_{Dice}
$$

The next cell implements the mathematical core. The expected result is a set of reusable metric and loss functions that work on PyTorch tensors.


In [ ]:
def dice_coefficient_from_logits(logits, targets, threshold=0.5, eps=1e-6):
    """Compute the hard Dice score from logits and binary targets."""
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()
    dims = tuple(range(1, preds.ndim))
    intersection = torch.sum(preds * targets, dim=dims)
    denominator = torch.sum(preds, dim=dims) + torch.sum(targets, dim=dims)
    dice = (2.0 * intersection + eps) / (denominator + eps)
    return dice.mean()


def soft_dice_loss_from_logits(logits, targets, eps=1e-6):
    """Compute differentiable Dice loss from logits and binary targets."""
    probs = torch.sigmoid(logits)
    dims = tuple(range(1, probs.ndim))
    intersection = torch.sum(probs * targets, dim=dims)
    denominator = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)
    dice = (2.0 * intersection + eps) / (denominator + eps)
    return 1.0 - dice.mean()


def combined_segmentation_loss(logits, targets):
    """Combine binary cross entropy with Dice loss."""
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    dice_loss = soft_dice_loss_from_logits(logits, targets)
    return bce + dice_loss


def numpy_binary_metrics(pred_mask, true_mask, eps=1e-6):
    """Compute common binary segmentation metrics using NumPy arrays."""
    pred = pred_mask.astype(bool)
    true = true_mask.astype(bool)
    tp = np.logical_and(pred, true).sum()
    fp = np.logical_and(pred, np.logical_not(true)).sum()
    fn = np.logical_and(np.logical_not(pred), true).sum()
    tn = np.logical_and(np.logical_not(pred), np.logical_not(true)).sum()
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    sensitivity = (tp + eps) / (tp + fn + eps)
    specificity = (tn + eps) / (tn + fp + eps)
    return {
        "dice": float(dice),
        "iou": float(iou),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
    }

# Quick sanity check on perfect and empty predictions.
tiny_true = np.array([[0, 1], [0, 1]], dtype=np.float32)
tiny_pred = tiny_true.copy()
print("Perfect prediction metrics:", numpy_binary_metrics(tiny_pred, tiny_true))


# 5. Synthetic Dataset Generator

Synthetic data keeps this notebook self-contained, avoids large downloads, and allows controlled experiments. It lets us vary lesion size, contrast, noise, blur, and prompt quality in a reproducible way. This is useful for teaching because every failure mode can be intentionally created and inspected.

The dataset below returns three input channels: image intensity, box prompt channel, and click prompt channel. The target is a binary lesion mask. The generator is deterministic because each index maps to a fixed random seed. The expected result is a small train/validation split that is ready for Colab-scale training.


In [ ]:
@dataclass
class SyntheticConfig:
    num_train: int = 64 if FAST_MODE else 256
    num_val: int = 16 if FAST_MODE else 64
    image_size: int = 64 if FAST_MODE else 96
    num_epochs: int = 5 if FAST_MODE else 10
    batch_size: int = 8 if FAST_MODE else 16
    learning_rate: float = 1e-3
    prompt_jitter: int = 3

config = SyntheticConfig()
print(config)


class SyntheticLesionPromptDataset(Dataset):
    """Small deterministic dataset of synthetic lesion images and prompts."""

    def __init__(
        self,
        num_samples,
        image_size=64,
        base_seed=0,
        prompt_jitter=3,
        contrast_range=(0.35, 0.75),
        noise_range=(0.04, 0.10),
        lesion_scale_range=(0.12, 0.25),
    ):
        self.num_samples = int(num_samples)
        self.image_size = int(image_size)
        self.base_seed = int(base_seed)
        self.prompt_jitter = int(prompt_jitter)
        self.contrast_range = contrast_range
        self.noise_range = noise_range
        self.lesion_scale_range = lesion_scale_range

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        rng = np.random.default_rng(self.base_seed + idx)
        contrast = rng.uniform(*self.contrast_range)
        noise_sigma = rng.uniform(*self.noise_range)
        lesion_scale = rng.uniform(*self.lesion_scale_range)
        sample = synthetic_lesion_sample(
            image_size=self.image_size,
            seed=self.base_seed + idx,
            prompt_jitter=self.prompt_jitter,
            contrast=contrast,
            noise_sigma=noise_sigma,
            lesion_scale=lesion_scale,
        )
        image = sample["image"]
        box_channel = sample["box_channel"]
        click_channel = sample["click_channel"]
        mask = sample["mask"]
        inputs = np.stack([image, box_channel, click_channel], axis=0).astype(np.float32)
        target = mask[None, :, :].astype(np.float32)
        return {
            "x": torch.from_numpy(inputs),
            "y": torch.from_numpy(target),
            "image": torch.from_numpy(image[None, :, :]),
            "box_channel": torch.from_numpy(box_channel[None, :, :]),
            "click_channel": torch.from_numpy(click_channel[None, :, :]),
            "box": torch.tensor(sample["box"], dtype=torch.long),
        }


train_dataset = SyntheticLesionPromptDataset(
    num_samples=config.num_train,
    image_size=config.image_size,
    base_seed=SEED,
    prompt_jitter=config.prompt_jitter,
)
val_dataset = SyntheticLesionPromptDataset(
    num_samples=config.num_val,
    image_size=config.image_size,
    base_seed=SEED + 10_000,
    prompt_jitter=config.prompt_jitter,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

batch = next(iter(train_loader))
print("Input batch shape:", tuple(batch["x"].shape))
print("Target batch shape:", tuple(batch["y"].shape))
assert batch["x"].shape[1] == 3, "Expected three input channels: image, box, click."
assert batch["y"].shape[1] == 1, "Expected one binary mask channel."


def visualize_dataset_samples(dataset, n=4):
    """Visualize image, prompt, and mask examples from a dataset."""
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    if n == 1:
        axes = axes[None, :]
    for row in range(n):
        item = dataset[row]
        image = item["x"][0].numpy()
        prompt = item["x"][1].numpy()
        mask = item["y"][0].numpy()
        box = tuple(item["box"].numpy().tolist())
        axes[row, 0].imshow(image, cmap="gray")
        draw_box(axes[row, 0], box)
        axes[row, 0].set_title("Image and prompt box")
        axes[row, 1].imshow(prompt, cmap="gray")
        axes[row, 1].set_title("Box prompt channel")
        axes[row, 2].imshow(mask, cmap="gray")
        axes[row, 2].set_title("Ground truth")
        for col in range(3):
            axes[row, col].axis("off")
    plt.tight_layout()
    plt.show()

visualize_dataset_samples(train_dataset, n=4)


# 6. Baseline Method

A useful baseline should be simple, reasonable, and imperfect. Here we use prompt-aware thresholding. The prompt box localizes the lesion, and the baseline thresholds intensities only inside that box. This is reasonable because many lesions are brighter or darker than local background. It may fail when contrast is low, noise is high, texture is misleading, or the prompt box misses part of the lesion.

This baseline teaches an important research habit: before training a model, compare against a simple method. If a learned model cannot beat a thoughtful baseline, the experiment needs more debugging or a stronger hypothesis.

The next cell implements the baseline and visualizes its output on a few validation examples.


In [ ]:
def prompt_threshold_baseline(image, box, closing=True):
    """Segment bright lesions using a threshold estimated inside the prompt box."""
    y0, x0, y1, x1 = [int(v) for v in box]
    roi = image[y0:y1 + 1, x0:x1 + 1]
    threshold = roi.mean() + 0.35 * roi.std()
    pred = np.zeros_like(image, dtype=np.float32)
    pred_roi = (roi >= threshold).astype(np.float32)
    if closing:
        pred_roi = binary_closing(pred_roi, structure=np.ones((3, 3))).astype(np.float32)
        pred_roi = binary_opening(pred_roi, structure=np.ones((2, 2))).astype(np.float32)
    pred[y0:y1 + 1, x0:x1 + 1] = pred_roi
    return pred


def visualize_baseline(dataset, n=4):
    """Show baseline outputs and error maps."""
    fig, axes = plt.subplots(n, 4, figsize=(12, 3 * n))
    if n == 1:
        axes = axes[None, :]
    for row in range(n):
        item = dataset[row]
        image = item["x"][0].numpy()
        mask = item["y"][0].numpy()
        box = tuple(item["box"].numpy().tolist())
        pred = prompt_threshold_baseline(image, box)
        error = np.abs(pred - mask)
        metrics = numpy_binary_metrics(pred, mask)
        axes[row, 0].imshow(image, cmap="gray")
        draw_box(axes[row, 0], box)
        axes[row, 0].set_title("Image and box")
        axes[row, 1].imshow(mask, cmap="gray")
        axes[row, 1].set_title("Ground truth")
        axes[row, 2].imshow(pred, cmap="gray")
        axes[row, 2].set_title(f"Baseline Dice={metrics['dice']:.2f}")
        axes[row, 3].imshow(error, cmap="magma")
        axes[row, 3].set_title("Absolute error")
        for col in range(4):
            axes[row, col].axis("off")
    plt.tight_layout()
    plt.show()

visualize_baseline(val_dataset, n=4)


# 7. Lightweight Learned or Research-Inspired Method

The main method is a tiny prompt-guided segmentation network. It is not MedSAM and does not use a transformer foundation model. Instead, it preserves the key research idea: the segmentation model receives both the image and the prompt, then predicts the target mask.

The model improves over the baseline by learning spatial and texture cues from examples. The prompt channels tell the model where to look; the convolutional layers learn how to refine boundaries inside and near the prompt. This connects to MedSAM-style systems because the prompt is treated as part of the model input, not as a post-processing rule.

What is simplified here:

- The images are synthetic and two-dimensional.
- The model is a tiny prompt-conditioned CNN, not a large vision transformer.
- The prompts are box and point channels, not a full prompt encoder.
- The training data is small and generated on the fly.

Full-scale research would require clinical-quality datasets, rigorous train/test separation, multiple modalities, external validation, uncertainty analysis, human factors evaluation, and compliance with institutional data governance.

The next cell defines the small model. The expected result is a compact network with far fewer parameters than research-scale models and a fast CPU-friendly forward pass.


In [ ]:
class TinyPromptSegNet(nn.Module):
    """A very small prompt-conditioned CNN for binary lesion segmentation."""

    def __init__(self, in_channels=3, hidden_channels=8):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.context = nn.Sequential(
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=2, dilation=2),
            nn.ReLU(inplace=True),
        )
        self.refine = nn.Sequential(
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, 1, kernel_size=1),
        )

    def forward(self, x):
        features = self.stem(x)
        features = self.context(features)
        logits = self.refine(features)
        return logits


model = TinyPromptSegNet(in_channels=3, hidden_channels=8).to(device)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", num_params)

with torch.no_grad():
    test_logits = model(batch["x"].to(device))
print("Model output shape:", tuple(test_logits.shape))
assert test_logits.shape == batch["y"].shape, "Model output must match target mask shape."


# 8. Lightweight Training Loop

The model optimizes a combination of binary cross entropy and Dice loss. Binary cross entropy supplies pixel-wise supervision, while Dice loss directly encourages foreground-overlap quality. Training is intentionally small so that every student can run the notebook in Colab without a paid GPU or large downloads.

GPU users can increase the dataset size, image size, or number of epochs after understanding the baseline behavior. Full-scale MedSAM-style training is not included because it requires large medical image-mask collections, careful validation, and substantial compute.

The next cell creates the model, optimizer, and training loop. It prints progress messages and plots the training loss and validation Dice curve. The expected result is that validation Dice should improve over the first few epochs, although small synthetic runs can vary.


In [ ]:
model = TinyPromptSegNet(in_channels=3, hidden_channels=8).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

history = {"train_loss": [], "val_dice": []}


def train_one_epoch(model, loader, optimizer, device):
    """Run one training epoch."""
    model.train()
    total_loss = 0.0
    for batch_idx, batch in enumerate(loader):
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = combined_segmentation_loss(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_val_dice(model, loader, device):
    """Compute mean validation Dice."""
    model.eval()
    dice_values = []
    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        logits = model(x)
        dice = dice_coefficient_from_logits(logits, y)
        dice_values.append(float(dice.detach().cpu()))
    return float(np.mean(dice_values))

print("Starting lightweight training...")
for epoch in range(1, config.num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_dice = evaluate_val_dice(model, val_loader, device)
    history["train_loss"].append(train_loss)
    history["val_dice"].append(val_dice)
    print(f"Epoch {epoch:02d}/{config.num_epochs} | train loss = {train_loss:.4f} | val Dice = {val_dice:.4f}")

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(history["train_loss"], marker="o", label="Training loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Training loss")
ax1.set_title("Lightweight training curve")
ax2 = ax1.twinx()
ax2.plot(history["val_dice"], marker="s", label="Validation Dice")
ax2.set_ylabel("Validation Dice")
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc="center right")
plt.show()


**Note for GPU users:** If you have access to a GPU, you can set `FAST_MODE = False`, increase `num_epochs`, increase the dataset size, or replace the tiny model with a stronger architecture. The current notebook keeps training small so that every student can run it in Colab.


# 9. Evaluation Metrics

A single metric rarely tells the whole story. Biomedical segmentation commonly uses overlap metrics and error-type metrics:

- Dice score measures foreground overlap and is useful for small objects.
- IoU measures intersection divided by union and is stricter than Dice.
- Sensitivity measures how much true foreground was found.
- Specificity measures how much true background was correctly rejected.

A model can have high specificity by predicting mostly background, so sensitivity and Dice are important when lesions are small. The next cell compares the prompt-aware baseline against the learned prompt-guided model on the validation set and displays a table and bar plot.


In [ ]:
@torch.no_grad()
def predict_model_mask(model, x_tensor, device, threshold=0.5):
    """Predict a binary mask for one sample tensor with shape [3, H, W]."""
    model.eval()
    logits = model(x_tensor[None].to(device))
    prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
    pred = (prob >= threshold).astype(np.float32)
    return pred, prob


def evaluate_methods(dataset, model, device):
    """Evaluate baseline and learned model on a dataset."""
    records = []
    for idx in range(len(dataset)):
        item = dataset[idx]
        image = item["x"][0].numpy()
        true_mask = item["y"][0].numpy()
        box = tuple(item["box"].numpy().tolist())
        baseline_pred = prompt_threshold_baseline(image, box)
        model_pred, _ = predict_model_mask(model, item["x"], device)
        for method_name, pred in [("Prompt threshold baseline", baseline_pred), ("Tiny prompt model", model_pred)]:
            metrics = numpy_binary_metrics(pred, true_mask)
            metrics["method"] = method_name
            metrics["sample_index"] = idx
            records.append(metrics)
    return records

records = evaluate_methods(val_dataset, model, device)
methods = sorted(set(r["method"] for r in records))
summary_rows = []
for method_name in methods:
    method_records = [r for r in records if r["method"] == method_name]
    row = {"method": method_name}
    for metric_name in ["dice", "iou", "sensitivity", "specificity"]:
        row[metric_name] = float(np.mean([r[metric_name] for r in method_records]))
    summary_rows.append(row)

if pd is not None:
    metrics_table = pd.DataFrame(summary_rows)
    display(metrics_table.round(3))
else:
    for row in summary_rows:
        print(row)

metric_names = ["dice", "iou", "sensitivity", "specificity"]
x_positions = np.arange(len(metric_names))
bar_width = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
for offset, row in zip([-bar_width / 2, bar_width / 2], summary_rows):
    values = [row[m] for m in metric_names]
    ax.bar(x_positions + offset, values, width=bar_width, label=row["method"])
ax.set_xticks(x_positions)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Mean metric value")
ax.set_title("Validation metric comparison")
ax.legend()
plt.show()


# 10. Visualization and Interpretation

Visualization should reveal not only whether a prediction is good, but how it fails. We will look at the input image, prompt box, ground truth, baseline prediction, learned prediction, error maps, and an uncertainty map.

For the learned model, uncertainty is approximated with Bernoulli entropy:

$$
H(p) = -p\log(p + \epsilon) - (1-p)\log(1-p + \epsilon)
$$

High entropy appears when the model probability is near 0.5. This is not a full uncertainty estimate, but it is a useful teaching tool for seeing where the model is unsure.

The next cell visualizes several validation examples. The expected result is that the learned method usually produces smoother, more lesion-shaped masks than thresholding, while still making mistakes under ambiguity.


In [ ]:
def probability_entropy(prob, eps=1e-6):
    """Compute normalized Bernoulli entropy for probabilities in [0, 1]."""
    entropy = -prob * np.log(prob + eps) - (1.0 - prob) * np.log(1.0 - prob + eps)
    return entropy / np.log(2.0)


def visualize_predictions(dataset, model, device, indices=(0, 1, 2)):
    """Visualize predictions, errors, and uncertainty for selected samples."""
    for idx in indices:
        item = dataset[idx]
        image = item["x"][0].numpy()
        true_mask = item["y"][0].numpy()
        box = tuple(item["box"].numpy().tolist())
        baseline_pred = prompt_threshold_baseline(image, box)
        model_pred, prob = predict_model_mask(model, item["x"], device)
        baseline_error = np.abs(baseline_pred - true_mask)
        model_error = np.abs(model_pred - true_mask)
        uncertainty = probability_entropy(prob)

        fig, axes = plt.subplots(1, 8, figsize=(20, 3))
        panels = [
            (image, "Input image", "gray"),
            (item["x"][1].numpy(), "Box prompt", "gray"),
            (true_mask, "Ground truth", "gray"),
            (baseline_pred, "Baseline pred", "gray"),
            (baseline_error, "Baseline error", "magma"),
            (model_pred, "Learned pred", "gray"),
            (model_error, "Learned error", "magma"),
            (uncertainty, "Model uncertainty", "viridis"),
        ]
        for ax, (img, title, cmap) in zip(axes, panels):
            ax.imshow(img, cmap=cmap)
            if title == "Input image":
                draw_box(ax, box)
            ax.set_title(title)
            ax.axis("off")
        plt.suptitle(f"Validation sample {idx}")
        plt.tight_layout()
        plt.show()

visualize_predictions(val_dataset, model, device, indices=(0, 1, 2))


## Interpretation

When the prompt is reasonably accurate and the lesion has visible contrast, the learned model should often recover a cleaner region than the threshold baseline. Error maps help identify false positives and false negatives. The uncertainty map should often be brighter near boundaries or ambiguous textured regions. If the learned model underperforms, inspect the training curve, dataset difficulty, prompt channels, and threshold used to convert probabilities into masks.


# 11. Ablation or Stress Test

Ablation matters because it tests which assumptions are supporting the method. Prompt-guided segmentation assumes that the prompt is informative. If the prompt is badly shifted or too loose, the model receives weaker localization information.

Here we vary prompt jitter while keeping the trained model fixed. This is a fast stress test: no retraining is performed. The expected trend is that both the baseline and the learned model may degrade as prompt quality worsens, but they may degrade in different ways.


In [ ]:
def evaluate_mean_dice_on_prompt_jitter(jitter_values, model, device, num_samples=16):
    """Evaluate mean Dice under different prompt jitter levels."""
    results = []
    for jitter in jitter_values:
        stress_dataset = SyntheticLesionPromptDataset(
            num_samples=num_samples,
            image_size=config.image_size,
            base_seed=SEED + 20_000 + jitter * 100,
            prompt_jitter=int(jitter),
        )
        records = evaluate_methods(stress_dataset, model, device)
        for method_name in sorted(set(r["method"] for r in records)):
            method_records = [r for r in records if r["method"] == method_name]
            mean_dice = float(np.mean([r["dice"] for r in method_records]))
            results.append({"prompt_jitter": jitter, "method": method_name, "mean_dice": mean_dice})
    return results

jitter_values = [0, 3, 6, 10, 14]
stress_results = evaluate_mean_dice_on_prompt_jitter(jitter_values, model, device, num_samples=12 if FAST_MODE else 32)

fig, ax = plt.subplots(figsize=(7, 4))
for method_name in sorted(set(r["method"] for r in stress_results)):
    rows = [r for r in stress_results if r["method"] == method_name]
    ax.plot([r["prompt_jitter"] for r in rows], [r["mean_dice"] for r in rows], marker="o", label=method_name)
ax.set_xlabel("Prompt jitter in pixels")
ax.set_ylabel("Mean Dice")
ax.set_ylim(0, 1.05)
ax.set_title("Stress test: prompt quality")
ax.legend()
plt.show()

if pd is not None:
    display(pd.DataFrame(stress_results).round(3))
else:
    for row in stress_results:
        print(row)


## Ablation Interpretation

If Dice decreases as prompt jitter increases, the method depends on prompt quality. This is not necessarily a flaw; it is part of the interaction contract. A promptable system should communicate when the prompt is unreliable, ask for refinement, or expose uncertainty. In real biomedical annotation tools, this motivates iterative prompting, human review, and active learning.


# 12. Failure Modes and Limitations

Failure analysis matters because average metrics can hide clinically important mistakes. A system may perform well on typical images and fail on rare but important cases. In biomedical settings, failures can arise from low contrast, severe noise, tiny targets, motion artifacts, prompt errors, unseen imaging protocols, or ambiguous boundaries.

The next cell creates a deliberately difficult case: low lesion contrast, high noise, and a poor prompt. The expected result is not guaranteed, but the case should be challenging enough to reveal limitations of both the baseline and the learned model.


In [ ]:
failure_sample = synthetic_lesion_sample(
    image_size=config.image_size,
    seed=SEED + 99_999,
    prompt_jitter=18,
    contrast=0.12,
    noise_sigma=0.18,
    lesion_scale=0.10,
    irregularity=0.20,
)

failure_x = np.stack(
    [failure_sample["image"], failure_sample["box_channel"], failure_sample["click_channel"]],
    axis=0,
).astype(np.float32)
failure_x_tensor = torch.from_numpy(failure_x)
true_mask = failure_sample["mask"]
box = failure_sample["box"]
baseline_pred = prompt_threshold_baseline(failure_sample["image"], box)
model_pred, model_prob = predict_model_mask(model, failure_x_tensor, device)
uncertainty = probability_entropy(model_prob)

baseline_metrics = numpy_binary_metrics(baseline_pred, true_mask)
model_metrics = numpy_binary_metrics(model_pred, true_mask)
print("Failure case baseline metrics:", baseline_metrics)
print("Failure case learned model metrics:", model_metrics)

fig, axes = plt.subplots(1, 7, figsize=(18, 3))
panels = [
    (failure_sample["image"], "Difficult image", "gray"),
    (failure_sample["box_channel"], "Bad box prompt", "gray"),
    (true_mask, "Ground truth", "gray"),
    (baseline_pred, "Baseline pred", "gray"),
    (model_pred, "Learned pred", "gray"),
    (np.abs(model_pred - true_mask), "Learned error", "magma"),
    (uncertainty, "Uncertainty", "viridis"),
]
for ax, (img, title, cmap) in zip(axes, panels):
    ax.imshow(img, cmap=cmap)
    if title == "Difficult image":
        draw_box(ax, box)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Failure Analysis

This failure case violates multiple assumptions at once. The lesion has weak contrast, the noise level is high, the lesion is small, and the prompt is inaccurate. The baseline may mistake noise for lesion. The learned model may either miss the lesion or over-trust the prompt region. A researcher might try better prompt refinement, uncertainty-aware prompting, contrast augmentation, harder synthetic training examples, calibration, iterative human feedback, or a stronger architecture trained on real data.


# 13. Optional Full-Scale Training Resources

This section does not run full-scale training. Full MedSAM-style training usually requires large datasets, careful preprocessing, institutional approval for clinical data, strong hardware, and rigorous external validation.

Useful resources:

- Original MedSAM repository: https://github.com/bowang-lab/MedSAM  
  Official code for MedSAM inference, training scripts, quickstart notebooks, GUI tools, and related resources.
- MedSAM paper: https://arxiv.org/abs/2304.12306  
  Describes the motivation, large-scale medical image-mask data, evaluation strategy, and foundation-model framing.
- Nature Communications article: https://www.nature.com/articles/s41467-024-44824-z  
  Peer-reviewed version of the MedSAM work.
- Segment Anything repository: https://github.com/facebookresearch/segment-anything  
  Official SAM implementation from Meta AI.
- Segment Anything paper: https://arxiv.org/abs/2304.02643  
  Foundation model for promptable natural-image segmentation.
- Medical Segmentation Decathlon: http://medicaldecathlon.com/  
  Public benchmark for medical segmentation research.
- MONAI: https://docs.monai.io/  
  Medical imaging deep learning toolkit.
- nnU-Net: https://github.com/MIC-DKFZ/nnUNet  
  Strong baseline and framework for biomedical segmentation.

The commented cell below shows what a full-scale setup might look like conceptually, but it is intentionally disabled.


In [ ]:
# Optional full-scale resources are intentionally not executed in this notebook.
# They may require large downloads, GPU memory, credentials, or institution-specific data governance.

# Example only. Do not run by default in Colab Free Tier.
# !git clone https://github.com/bowang-lab/MedSAM
# %cd MedSAM
# !pip install -e .
# After reviewing licenses and data rules, follow the official repository instructions.


# 14. Research Extension Mini-Project

## Hypothesis

A prompt-guided segmentation model trained with prompt perturbation augmentation will be more robust to inaccurate prompts than a model trained only with tight prompts.

## What to Modify

Train two tiny models:

1. Model A with `prompt_jitter = 0`.
2. Model B with `prompt_jitter` sampled from a larger range, such as 0 to 10 pixels.

Keep the dataset size, architecture, optimizer, and number of epochs fixed.

## What to Measure

Measure Dice, IoU, sensitivity, and specificity across a stress-test suite with prompt jitter values such as 0, 3, 6, 10, and 14 pixels.

## Expected Result

Model B should be more robust to inaccurate prompts, especially at higher jitter levels, because it has seen prompt noise during training.

## Possible Negative Result

Model B may perform worse on accurate prompts if augmentation makes the learning problem too hard for a tiny model. It may also fail if synthetic images are too simple or too noisy.

## How to Report Findings

Report the training settings, parameter count, dataset size, random seed, metric table, stress-test plot, representative success cases, representative failure cases, and one paragraph explaining whether the hypothesis was supported.

## Is a GPU Recommended?

A GPU is helpful but not required. For a stronger mini-project with multiple training runs, a GPU is recommended.


# 15. Exercises and Reflection Questions

## Part A: Conceptual Understanding

1. What problem does prompt-guided segmentation solve compared with fully automatic segmentation?
2. Why can a bounding box prompt improve segmentation but still fail to determine the correct boundary?
3. Why is Dice score often useful for biomedical lesion segmentation?
4. What assumptions does this notebook make about lesion appearance?
5. Why is synthetic data useful for controlled experiments but insufficient for clinical claims?

## Part B: Implementation Understanding

1. What are the three input channels to the tiny prompt-guided model?
2. Where in the code is the point prompt represented?
3. Why does the training loss combine binary cross entropy and Dice loss?
4. What does `FAST_MODE = True` control?
5. How would you change the code to train on darker lesions instead of brighter lesions?
6. Which function would you modify to create more irregular lesion boundaries?
7. Why does validation use `torch.no_grad()`?

## Part C: Research Thinking

1. How would you evaluate whether a promptable segmentation model is clinically useful?
2. What failure cases would be unacceptable in a real biomedical annotation tool?
3. How might uncertainty maps support human-in-the-loop annotation?
4. How could an AutoBio laboratory automation system use promptable segmentation?
5. What data governance, privacy, and validation issues arise when moving from synthetic to clinical data?
6. What would be a fair baseline against a full MedSAM model?
7. How would you design an experiment to compare point prompts, box prompts, and combined prompts?


# 16. Summary and Further Reading

In this notebook, you learned how prompt-guided biomedical segmentation works at a conceptual and practical level. You implemented a deterministic synthetic lesion generator, a prompt-aware thresholding baseline, a tiny prompt-guided CNN, Dice and IoU-style metrics, a lightweight training loop, validation curves, prediction visualizations, uncertainty maps, prompt-quality stress tests, and failure-mode analysis.

You lightly trained a small model on synthetic data using `FAST_MODE = True`, automatic CPU/GPU selection, and no large downloads. You visualized how prompts guide segmentation and measured whether a learned model improves over a baseline.

This method matters because promptable segmentation can reduce annotation burden, support human-in-the-loop analysis, and help biomedical researchers interactively delineate objects of interest. Use it when a user can provide a meaningful prompt and when segmentation boundaries are visible enough for a model to infer. Do not rely on it without validation when prompts are inaccurate, targets are ambiguous, imaging protocols shift, or errors could affect clinical decisions.

Study next:

- MedSAM official repository: https://github.com/bowang-lab/MedSAM
- MedSAM paper: https://arxiv.org/abs/2304.12306
- Segment Anything paper: https://arxiv.org/abs/2304.02643
- MONAI tutorials: https://monai.io/
- Medical Segmentation Decathlon: http://medicaldecathlon.com/
- nnU-Net: https://github.com/MIC-DKFZ/nnUNet

The bridge from this notebook to research is not merely scaling the model. It is designing careful prompts, stress tests, external validation, uncertainty analysis, human-centered workflows, and clinically meaningful evaluations.
